# Demo 2
## Controlled Pendulum
### Hybrid: FMPy Co-Simulation & OpenSim Pendulum Model

**Modelica Models**

In [2]:
from path import Path

root = Path(r'/home/flo/repos/SystemSimulation/demos/ControlledPendulum/')
pkg_dir = Path(root / "ControlledPendulum")
pkg_str = str(root / "ControlledPendulum" / "package.mo")
pkg_name = "ControlledPendulum"

model_names = Path(pkg_dir).files("*.mo")
model_names = [m.stem for m in model_names if m.stem != "package"]

for model_name in model_names:
    print(f"Found model: {model_name}")

Found model: Demo_Driven
Found model: Demo_UndrivenWithWall
Found model: Drive
Found model: PID_Continuous
Found model: Pendulum
Found model: ImpactWall
Found model: Demo_UndrivenWallDiscrete
Found model: AngleEncoder
Found model: Demo_DrivenWithWall
Found model: PID_Sampled
Found model: Reference


**Helper Functions for FMU Creation**

In [3]:
from OMPython import ModelicaSystem
import shutil

def create_modelica_system(model_name):
    model = ModelicaSystem(pkg_str, model_name, verbose=True)
    return model

def create_fmu(model: ModelicaSystem, fmuType="cs"):
    model.buildModel()
    fmu_path = model.convertMo2Fmu(fmuType=fmuType)
    return fmu_path

def move_file(src_path, dst_path):
    shutil.move(src_path, dst_path)

**Create FMUs for the Models**

In [13]:
# Check if fmus exist, if not create them
fmu_dir = root / "FMUs"
if not fmu_dir.exists():
    fmu_dir.mkdir()
fmu_path_dict = {}

for model_name in model_names:
    fmu_path = fmu_dir / f"{model_name}.fmu"
    if not fmu_path.exists():
        print(f"Creating FMU for model: {model_name}")
        model = create_modelica_system(pkg_name + "." + model_name)
        temp_fmu_path = create_fmu(model)
        move_file(temp_fmu_path, fmu_path)
    else:
        print(f"FMU already exists for model: {model_name}")
    fmu_path_dict[model_name] = str(fmu_path)

FMU already exists for model: Demo_Driven
FMU already exists for model: Demo_UndrivenWithWall
FMU already exists for model: Drive
FMU already exists for model: PID_Continuous
FMU already exists for model: Pendulum
FMU already exists for model: ImpactWall
FMU already exists for model: Demo_UndrivenWallDiscrete
FMU already exists for model: AngleEncoder
FMU already exists for model: Demo_DrivenWithWall
FMU already exists for model: PID_Sampled
FMU already exists for model: Reference


In [14]:
# Import FMPy
from fmpy import read_model_description

fmu_dict = {}

# Read model descriptions and variable references
for model_name, fmu_path in fmu_path_dict.items():
    print(30 * "-")
    print(f"Reading FMU: {model_name}")
    fmu_dict[model_name] = {}
    fmu_dict[model_name]["ModelDescription"] = read_model_description(fmu_path)
    vrs = {}
    print("  Variables:")
    for variable in fmu_dict[model_name]["ModelDescription"].modelVariables:
        vrs[variable.name] = variable.valueReference
        print(f"    {variable.name:<30} : {variable.valueReference}")

    fmu_dict[model_name]["VariableReferences"] = vrs

------------------------------
Reading FMU: Demo_Driven
  Variables:
    drive.I                        : 0
    pendulum.omega_state           : 1
    pendulum.q_state               : 2
    pid.D.x                        : 3
    pid.I.y                        : 4
    der(drive.I)                   : 5
    der(pendulum.omega_state)      : 6
    der(pendulum.q_state)          : 7
    der(pid.D.x)                   : 8
    der(pid.I.y)                   : 9
    drive.torque                   : 14
    pid.u                          : 21
    reference.q_ref                : 22
    sensor_ref.U_q                 : 24
    sensor_state.U_q               : 27
    drive.U_M                      : 29
    pendulum.L                     : 30
    pendulum.m                     : 31
    pendulum.omega0                : 32
    pendulum.q0                    : 33
    pid.Nd                         : 40
    pid.Td                         : 42
    pid.Ti                         : 43
    pid.k            

In [20]:
# Dictionary to hold variable references
vars_rf = {}
vars_rf['q_ref'] = fmu_dict['Reference']['VariableReferences']['q_ref']

vars_rf['q_state'] = fmu_dict['Pendulum']['VariableReferences']['q_state']
vars_rf['omega_state'] = fmu_dict['Pendulum']['VariableReferences']['omega_state']
vars_rf['torque'] = fmu_dict['Pendulum']['VariableReferences']['torque']

vars_rf['q_sensor'] = fmu_dict['AngleEncoder']['VariableReferences']['q']
vars_rf['U_q'] = fmu_dict['AngleEncoder']['VariableReferences']['U_q']

vars_rf['pid_y'] = fmu_dict['PID_Continuous']['VariableReferences']['y']
vars_rf['pid_ref'] = fmu_dict['PID_Continuous']['VariableReferences']['ref']
vars_rf['pid_u'] = fmu_dict['PID_Continuous']['VariableReferences']['u']

vars_rf['drive_u'] = fmu_dict['Drive']['VariableReferences']['u_control']
vars_rf['drive_omega'] = fmu_dict['Drive']['VariableReferences']['omega']
vars_rf['drive_torque'] = fmu_dict['Drive']['VariableReferences']['torque']

In [21]:
from fmpy import extract
from fmpy.fmi2 import FMU2Slave

# Instantiate Reference Trajectory
unzip_dir = extract(fmu_path_dict['Reference'])
ref_fmu = FMU2Slave(guid=fmu_dict['Reference']["ModelDescription"].guid,
                    unzipDirectory=unzip_dir,
                    modelIdentifier=fmu_dict['Reference']["ModelDescription"].coSimulation.modelIdentifier,
                    instanceName='ref_fmu')
ref_fmu.instantiate()

# Instantiate Sensors for Reference Trajectory and Pendulum
unzip_dir = extract(fmu_path_dict['AngleEncoder'])
sensor_ref_fmu = FMU2Slave(guid=fmu_dict['AngleEncoder']["ModelDescription"].guid,
                         unzipDirectory=unzip_dir,
                         modelIdentifier=fmu_dict['AngleEncoder']["ModelDescription"].coSimulation.modelIdentifier,
                         instanceName='sensor_ref_fmu')
sensor_ref_fmu.instantiate()

sensor_state_fmu = FMU2Slave(guid=fmu_dict['AngleEncoder']["ModelDescription"].guid,
                         unzipDirectory=unzip_dir,
                         modelIdentifier=fmu_dict['AngleEncoder']["ModelDescription"].coSimulation.modelIdentifier,
                         instanceName='sensor_state_fmu')
sensor_state_fmu.instantiate()

# Instantiate Controller
unzip_dir = extract(fmu_path_dict['PID_Continuous'])
pid_fmu = FMU2Slave(guid=fmu_dict['PID_Continuous']["ModelDescription"].guid,
                    unzipDirectory=unzip_dir,
                    modelIdentifier=fmu_dict['PID_Continuous']["ModelDescription"].coSimulation.modelIdentifier,
                    instanceName='pid_fmu')
pid_fmu.instantiate()

# Instantiate Drive
unzip_dir = extract(fmu_path_dict['Drive'])
drive = FMU2Slave(guid=fmu_dict['Drive']["ModelDescription"].guid,
                  unzipDirectory=unzip_dir,
                  modelIdentifier=fmu_dict['Drive']["ModelDescription"].coSimulation.modelIdentifier,
                  instanceName='drive')
drive.instantiate()

fmu_list = [ref_fmu, sensor_ref_fmu, sensor_state_fmu, pid_fmu, drive]

In [22]:
from OpenSim.opensim_pendulum import PendulumModel

pendulum_model = PendulumModel()

In [23]:
t = 0.0
tf = 10.0
h = 0.001

for fmu in fmu_list:
    fmu.reset()
    fmu.setupExperiment(startTime=t)
    fmu.enterInitializationMode()
    fmu.exitInitializationMode()

# Initialize logging arrays
ts = []
q_ref_log, q_state_log, omega_state_log = [], [], []
U_ref_log, U_state_log = [], []

In [24]:
while t < tf:
    # 1) Read the current reference position
    ref_fmu.doStep(t, h)
    q_ref = ref_fmu.getReal([vars_rf['q_ref']])
    
    # 2) Read the plant state at the current time
    q_state = [pendulum_model.get_q()]
    omega_state = [pendulum_model.get_omega()]

    # 3) Sensors: set inputs -> step -> read outputs
    sensor_ref_fmu.setReal([vars_rf['q_sensor']], q_ref)
    sensor_state_fmu.setReal([vars_rf['q_sensor']], q_state)
    sensor_ref_fmu.doStep(t, h)
    sensor_state_fmu.doStep(t, h)
    U_q_ref = sensor_ref_fmu.getReal([vars_rf['U_q']])
    U_q_state = sensor_state_fmu.getReal([vars_rf['U_q']])

    # 4) Controller: set inputs -> step -> read outputs
    pid_fmu.setReal([vars_rf['pid_y'], vars_rf['pid_ref']],
                    [U_q_state[0], U_q_ref[0]])
    pid_fmu.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    u_pid = pid_fmu.getReal([vars_rf['pid_u']])

    # 5) Drive: set inputs -> step -> read outputs
    drive.setReal([vars_rf['drive_u'], vars_rf['drive_omega']],
                  [u_pid[0], omega_state[0]])
    drive.doStep(currentCommunicationPoint=t, communicationStepSize=h)
    torque = drive.getReal([vars_rf['drive_torque']])[0]

    # 6) Plant: set inputs -> step
    pendulum_model.set_torque(torque)
    pendulum_model.do_step(t, h)

    # Log data
    ts.append(t)
    q_ref_log.append(q_ref[0])
    q_state_log.append(q_state[0])
    omega_state_log.append(omega_state[0])
    U_ref_log.append(U_q_ref[0])
    U_state_log.append(U_q_state[0])
    
    # Advance time
    t += h

In [25]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts, y=q_ref_log, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=ts, y=q_state_log, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()

In [26]:
from opensim import TimeSeriesTable, STOFileAdapter, RowVector
table = TimeSeriesTable()
table.setColumnLabels(['q', 'q_dot'])

for i in range(len(ts)):
    row = RowVector(2)
    row[0] = q_state_log[i]
    row[1] = omega_state_log[i]

    
    # appendRow takes time as first argument, then the row data
    table.appendRow(ts[i], row)

# Use STOFileAdapter to write the table
adapter = STOFileAdapter()
adapter.write(table, r'OpenSim/demo_hybrid_results.sto')